# PatchCore Training Notebook (Pretrained ResNet18 Backbone)

This notebook runs a PatchCore sweep on the shared `64x64` 5% test-defect split using a frozen ImageNet-pretrained `ResNet18` backbone.

Workflow:
- load the shared processed wafer split
- build a frozen pretrained `ResNet18` PatchCore backbone
- sweep a few memory-bank / reduction settings
- save each variant to its own artifact folder
- compare variants in one summary table
- carry one selected variant into downstream plots and failure analysis

This edited version also automatically:
- saves score histograms for each variant
- extracts validation/test embeddings for the selected PatchCore model
- generates UMAP plots (split-colored and score-colored)
- writes plot/image artifacts into each variant's `evaluation/plots/` folder

In [ ]:
from pathlib import Path
import copy
import json
import random
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from torch.utils.data import DataLoader

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / "src" / "wafer_defect").exists() and (candidate / "configs").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root containing src/wafer_defect and configs/")

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from wafer_defect.config import load_toml
from wafer_defect.data.wm811k import WaferMapDataset
from wafer_defect.evaluation import export_reference_umap_bundle, summarize_threshold_metrics, sweep_threshold_metrics
from wafer_defect.models.patchcore import PatchCoreModel
from wafer_defect.training.patchcore import build_memory_subset, collect_memory_bank

In [ ]:
CONFIG_PATH = REPO_ROOT / "configs" / "training" / "train_patchcore_resnet18.toml"
config = load_toml(CONFIG_PATH)

PATCHCORE_SWEEP = [
    {"name": "mean_mb10k", "memory_bank_size": 10_000, "reduction": "mean", "topk_ratio": 0.10},
    {"name": "mean_mb50k", "memory_bank_size": 50_000, "reduction": "mean", "topk_ratio": 0.10},
    {"name": "topk_mb10k_r005", "memory_bank_size": 10_000, "reduction": "topk_mean", "topk_ratio": 0.05},
    {"name": "topk_mb50k_r005", "memory_bank_size": 50_000, "reduction": "topk_mean", "topk_ratio": 0.05},
    {"name": "topk_mb50k_r010", "memory_bank_size": 50_000, "reduction": "topk_mean", "topk_ratio": 0.10},
    {"name": "max_mb50k", "memory_bank_size": 50_000, "reduction": "max", "topk_ratio": 0.10},
]

SELECTED_VARIANT_NAME = None
AUTO_SELECT_METRIC = "f1"

# Optional quick smoke-run overrides:
# config["data"]["metadata_csv"] = "data/processed/x64/wm811k/metadata_dev.csv"
# config["run"]["output_dir"] = "experiments/anomaly_detection/patchcore/resnet18/x64/main/artifacts/patchcore_resnet18_dev"
# PATCHCORE_SWEEP = PATCHCORE_SWEEP[:2]

base_output_dir = REPO_ROOT / config["run"]["output_dir"]
config

# Visualization / artifact settings
GENERATE_VARIANT_HISTOGRAMS = True
GENERATE_SELECTED_UMAP = True
UMAP_RANDOM_STATE = 42
UMAP_MAX_TRAIN_REFERENCE = 5000
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.10
UMAP_MAX_VAL_NORMAL = 4000
UMAP_MAX_TEST_NORMAL = 8000
UMAP_MAX_TEST_ANOMALY = 4000

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def resolve_device(device_name: str) -> torch.device:
    if device_name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(device_name)

seed = int(config["run"]["seed"])
set_seed(seed)
device = resolve_device(config["training"].get("device", "auto"))
device

In [ ]:
image_size = int(config["data"].get("image_size", 64))
batch_size = int(config["data"].get("batch_size", 64))
num_workers = int(config["data"].get("num_workers", 0))
metadata_path = REPO_ROOT / config["data"]["metadata_csv"]
metadata = pd.read_csv(metadata_path)

display(metadata.head())
display(metadata["split"].value_counts().rename_axis("split").to_frame("count"))
display(metadata["is_anomaly"].value_counts().rename_axis("is_anomaly").to_frame("count"))

train_dataset = WaferMapDataset(metadata_path, split="train", image_size=image_size)
val_dataset = WaferMapDataset(metadata_path, split="val", image_size=image_size)
test_dataset = WaferMapDataset(metadata_path, split="test", image_size=image_size)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print(f"train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")

In [ ]:
base_model_config = config["model"]
memory_bank_cache: dict[int, dict[str, object]] = {}

def build_patchcore_model(*, reduction: str, topk_ratio: float) -> PatchCoreModel:
    return PatchCoreModel(
        image_size=image_size,
        backbone_type=str(base_model_config.get("backbone_type", "resnet18")),
        use_batchnorm=bool(base_model_config.get("use_batchnorm", True)),
        pretrained=bool(base_model_config.get("pretrained", True)),
        freeze_backbone=bool(base_model_config.get("freeze_backbone", True)),
        backbone_input_size=int(base_model_config.get("backbone_input_size", 224)),
        normalize_imagenet=bool(base_model_config.get("normalize_imagenet", True)),
        reduction=str(reduction),
        topk_ratio=float(topk_ratio),
        query_chunk_size=int(base_model_config.get("query_chunk_size", 2048)),
        memory_chunk_size=int(base_model_config.get("memory_chunk_size", 8192)),
    ).to(device)

def get_memory_bank_info(memory_bank_size: int) -> dict[str, object]:
    if memory_bank_size not in memory_bank_cache:
        temp_model = build_patchcore_model(reduction="mean", topk_ratio=0.10)
        memory_subset = build_memory_subset(
            train_dataset,
            memory_bank_size=memory_bank_size,
            patches_per_image=temp_model.patches_per_image,
            seed=seed,
        )
        memory_loader = DataLoader(memory_subset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
        memory_bank = collect_memory_bank(
            model=temp_model,
            dataloader=memory_loader,
            device=device,
            target_size=memory_bank_size,
            seed=seed,
        )
        memory_bank_cache[memory_bank_size] = {
            "memory_bank": memory_bank.cpu(),
            "memory_subset_images": len(memory_subset),
            "patches_per_image": int(temp_model.patches_per_image),
            "feature_dim": int(temp_model.feature_dim),
        }
        del temp_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return memory_bank_cache[memory_bank_size]

def collect_scores(model: PatchCoreModel, dataloader: DataLoader) -> pd.DataFrame:
    rows = []
    model.eval()
    with torch.inference_mode():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            scores = model(inputs).cpu().numpy()
            labels = labels.cpu().numpy()
            for score, label in zip(scores, labels):
                rows.append({"score": float(score), "is_anomaly": int(label)})
    return pd.DataFrame(rows)

print({
    "sweep_variants": [variant["name"] for variant in PATCHCORE_SWEEP],
    "base_output_dir": str(base_output_dir),
    "backbone_type": str(base_model_config.get("backbone_type", "resnet18")),
    "pretrained": bool(base_model_config.get("pretrained", True)),
})

In [ ]:
def _import_umap():
    try:
        import umap.umap_ as umap
        return umap
    except Exception as exc:
        raise ImportError(
            "UMAP is required for embedding visualization. Install it with `pip install umap-learn`."
        ) from exc

def get_patchcore_embeddings(model, inputs):
    if hasattr(model, "patch_embeddings") and callable(model.patch_embeddings):
        out = model.patch_embeddings(inputs)
    elif hasattr(model, "feature_map") and callable(model.feature_map):
        out = model.feature_map(inputs)
    else:
        raise AttributeError("PatchCore model missing embedding method")

    if isinstance(out, (tuple, list)):
        for item in out:
            if hasattr(item, "shape"):
                out = item
                break

    if out.dim() == 3:
        out = out.mean(dim=1)
    elif out.dim() > 3:
        out = out.view(out.size(0), -1)

    return out

def collect_patchcore_embeddings(model: PatchCoreModel, dataloader: DataLoader) -> tuple[np.ndarray, np.ndarray]:
    all_embeddings = []
    all_labels = []
    model.eval()
    with torch.inference_mode():
        for inputs, labels in dataloader:
            device = next(model.parameters()).device
            inputs = inputs.to(device)
            embeddings = get_patchcore_embeddings(model, inputs)
            embeddings = embeddings.view(embeddings.size(0), -1).detach().cpu().numpy()
            all_embeddings.append(embeddings)
            all_labels.append(labels.cpu().numpy())
    return np.concatenate(all_embeddings, axis=0), np.concatenate(all_labels, axis=0)

def save_score_histograms(
    *,
    val_scores_df: pd.DataFrame,
    test_scores_df: pd.DataFrame,
    threshold_sweep_df: pd.DataFrame,
    threshold: float,
    best_sweep_threshold: float,
    variant_name: str,
    plots_dir: Path,
) -> None:
    plots_dir.mkdir(parents=True, exist_ok=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(val_scores_df["score"], bins=40, alpha=0.85, color="#4d908e")
    axes[0].axvline(threshold, color="red", linestyle="--", label=f"95th pct = {threshold:.4f}")
    axes[0].set_title(f"Validation Score Distribution\n{variant_name}")
    axes[0].set_xlabel("Anomaly score")
    axes[0].set_ylabel("Count")
    axes[0].legend()

    axes[1].hist(
        test_scores_df.loc[test_scores_df["is_anomaly"] == 0, "score"],
        bins=50,
        alpha=0.6,
        label=f"Test normal ({(test_scores_df['is_anomaly'] == 0).sum()})",
        density=True,
    )
    axes[1].hist(
        test_scores_df.loc[test_scores_df["is_anomaly"] == 1, "score"],
        bins=50,
        alpha=0.6,
        label=f"Test anomaly ({(test_scores_df['is_anomaly'] == 1).sum()})",
        density=True,
    )
    axes[1].axvline(threshold, color="#277da1", linestyle="--", label=f"95th pct = {threshold:.4f}")
    axes[1].axvline(best_sweep_threshold, color="#1d3557", linestyle="-.", label=f"Best sweep = {best_sweep_threshold:.4f}")
    axes[1].set_title(f"Test Score Distribution\n{variant_name}")
    axes[1].set_xlabel("Anomaly score")
    axes[1].set_ylabel("Density")
    axes[1].legend()

    plt.tight_layout()
    fig.savefig(plots_dir / "score_histogram.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(threshold_sweep_df["threshold"], threshold_sweep_df["precision"], label="precision")
    ax.plot(threshold_sweep_df["threshold"], threshold_sweep_df["recall"], label="recall")
    ax.plot(threshold_sweep_df["threshold"], threshold_sweep_df["f1"], label="f1")
    ax.axvline(threshold, color="#277da1", linestyle="--", label=f"95th pct = {threshold:.4f}")
    ax.axvline(best_sweep_threshold, color="#1d3557", linestyle="-.", label=f"Best sweep = {best_sweep_threshold:.4f}")
    ax.set_title(f"Threshold Sweep\n{variant_name}")
    ax.set_xlabel("Threshold")
    ax.set_ylabel("Metric value")
    ax.legend()
    plt.tight_layout()
    fig.savefig(plots_dir / "threshold_sweep_metrics.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

def save_umap_artifacts(
    *,
    variant_name: str,
    plots_dir: Path,
    train_embeddings: np.ndarray,
    val_embeddings: np.ndarray,
    val_labels: np.ndarray,
    test_embeddings: np.ndarray,
    test_labels: np.ndarray,
    val_scores: np.ndarray,
    test_scores: np.ndarray,
) -> pd.DataFrame:
    umap = _import_umap()
    bundle = export_reference_umap_bundle(
        output_dir=plots_dir,
        umap_module=umap,
        train_normal_embeddings=train_embeddings,
        val_embeddings=val_embeddings,
        val_labels=val_labels,
        test_embeddings=test_embeddings,
        test_labels=test_labels,
        val_model_scores=val_scores.astype(np.float32),
        test_model_scores=test_scores.astype(np.float32),
        threshold_quantile=0.95,
        random_state=UMAP_RANDOM_STATE,
        pca_components=50,
        n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        knn_k=15,
        metric="euclidean",
        max_train_reference=UMAP_MAX_TRAIN_REFERENCE,
        max_val_normal=UMAP_MAX_VAL_NORMAL,
        max_test_normal=UMAP_MAX_TEST_NORMAL,
        max_test_anomaly=UMAP_MAX_TEST_ANOMALY,
        title_prefix=f"Reference-Fit PatchCore Embeddings\n{variant_name}",
        points_filename="embedding_umap_points.csv",
        split_plot_filename="umap_by_split.png",
        score_plot_filename="umap_by_score.png",
        summary_filename="umap_summary.json",
        sweep_filename="umap_knn_threshold_sweep.csv",
    )
    return bundle["points_df"]

def save_patchcore_variant_artifacts(
    *,
    model: PatchCoreModel,
    variant_name: str,
    eval_dir: Path,
    val_scores_df: pd.DataFrame,
    test_scores_df: pd.DataFrame,
    threshold_sweep_df: pd.DataFrame,
    threshold: float,
    best_sweep_threshold: float,
    include_umap: bool,
) -> dict[str, object]:
    plots_dir = eval_dir / "plots"
    plots_dir.mkdir(parents=True, exist_ok=True)

    save_score_histograms(
        val_scores_df=val_scores_df,
        test_scores_df=test_scores_df,
        threshold_sweep_df=threshold_sweep_df,
        threshold=threshold,
        best_sweep_threshold=best_sweep_threshold,
        variant_name=variant_name,
        plots_dir=plots_dir,
    )

    artifact_info = {"plots_dir": plots_dir, "umap_points_path": None}

    if include_umap:
        train_embeddings, train_labels = collect_patchcore_embeddings(model, train_loader)
        val_embeddings, val_labels = collect_patchcore_embeddings(model, val_loader)
        test_embeddings, test_labels = collect_patchcore_embeddings(model, test_loader)

        np.save(eval_dir / "train_embeddings.npy", train_embeddings)
        np.save(eval_dir / "val_embeddings.npy", val_embeddings)
        np.save(eval_dir / "test_embeddings.npy", test_embeddings)
        np.save(eval_dir / "train_labels.npy", train_labels)
        np.save(eval_dir / "val_labels.npy", val_labels)
        np.save(eval_dir / "test_labels.npy", test_labels)
        np.save(eval_dir / "val_scores.npy", val_scores_df["score"].to_numpy(dtype=np.float32))
        np.save(eval_dir / "test_scores.npy", test_scores_df["score"].to_numpy(dtype=np.float32))

        umap_df = save_umap_artifacts(
            variant_name=variant_name,
            plots_dir=plots_dir,
            train_embeddings=train_embeddings,
            val_embeddings=val_embeddings,
            val_labels=val_labels,
            test_embeddings=test_embeddings,
            test_labels=test_labels,
            val_scores=val_scores_df["score"].to_numpy(dtype=np.float32),
            test_scores=test_scores_df["score"].to_numpy(dtype=np.float32),
        )
        artifact_info["umap_points_path"] = plots_dir / "embedding_umap_points.csv"
        artifact_info["umap_rows"] = len(umap_df)

    return artifact_info

# Artifact reload helpers for fresh-kernel UMAP regeneration.
def resolve_selected_variant_name_from_artifacts() -> str:
    if SELECTED_VARIANT_NAME is not None:
        return str(SELECTED_VARIANT_NAME)

    sweep_summary_path = base_output_dir / "patchcore_sweep_summary.json"
    if sweep_summary_path.exists():
        with sweep_summary_path.open("r", encoding="utf-8") as handle:
            sweep_summary = json.load(handle)
        selected_name = sweep_summary.get("selected_variant_name")
        if selected_name:
            return str(selected_name)

    sweep_results_path = base_output_dir / "patchcore_sweep_results.csv"
    if sweep_results_path.exists():
        sweep_results_df = pd.read_csv(sweep_results_path)
        if sweep_results_df.empty:
            raise FileNotFoundError(
                f"Found {sweep_results_path}, but it is empty. Rerun the full sweep in cell 8."
            )
        ranking_metrics = [AUTO_SELECT_METRIC, *[metric for metric in ["f1", "auroc", "auprc"] if metric != AUTO_SELECT_METRIC]]
        sort_columns = [column for column in ranking_metrics if column in sweep_results_df.columns]
        if sort_columns:
            sweep_results_df = sweep_results_df.sort_values(sort_columns, ascending=False)
        return str(sweep_results_df.iloc[0]["name"])

    raise FileNotFoundError(
        "No saved sweep summary/results found under "
        f"{base_output_dir}. Rerun the full sweep in cell 8 first."
    )

def load_selected_variant_from_artifacts(variant_name: str) -> dict[str, object]:
    run_output_dir = base_output_dir / variant_name
    eval_dir = run_output_dir / "evaluation"
    summary_path = run_output_dir / "summary.json"
    checkpoint_path = run_output_dir / "best_model.pt"
    val_scores_path = eval_dir / "val_scores.csv"
    test_scores_path = eval_dir / "test_scores.csv"
    threshold_sweep_path = eval_dir / "threshold_sweep.csv"

    required_paths = [
        summary_path,
        checkpoint_path,
        val_scores_path,
        test_scores_path,
        threshold_sweep_path,
    ]
    missing_paths = [str(item) for item in required_paths if not item.exists()]
    if missing_paths:
        missing_message = "\n".join(missing_paths)
        raise FileNotFoundError(
            "Missing saved artifacts for fresh-kernel reload. "
            "Rerun the full sweep in cell 8 if you want to regenerate them:\n"
            f"{missing_message}"
        )

    with summary_path.open("r", encoding="utf-8") as handle:
        summary = json.load(handle)

    val_scores_df = pd.read_csv(val_scores_path)
    test_scores_df = pd.read_csv(test_scores_path)
    threshold_sweep_df = pd.read_csv(threshold_sweep_path)

    threshold = float(summary["threshold"])
    labels = test_scores_df["is_anomaly"].to_numpy()
    scores = test_scores_df["score"].to_numpy()
    metrics = summarize_threshold_metrics(labels, scores, threshold)

    if "f1" in threshold_sweep_df.columns:
        best_sweep_row = threshold_sweep_df.sort_values("f1", ascending=False).iloc[0]
        best_sweep = {
            key: float(value) if key in {"threshold", "precision", "recall", "f1"} else value
            for key, value in best_sweep_row.to_dict().items()
        }
    else:
        best_sweep = {
            "threshold": threshold,
            "precision": float(metrics["precision"]),
            "recall": float(metrics["recall"]),
            "f1": float(metrics["f1"]),
        }

    checkpoint = torch.load(checkpoint_path, map_location=device)
    reduction = str(summary.get("reduction", base_model_config.get("reduction", "mean")))
    topk_ratio = float(summary.get("topk_ratio", base_model_config.get("topk_ratio", 0.10)))
    model = build_patchcore_model(reduction=reduction, topk_ratio=topk_ratio)

    state_dict = checkpoint["model_state_dict"]
    memory_bank = state_dict.get("memory_bank")
    if memory_bank is None:
        raise KeyError(
            f"Checkpoint at {checkpoint_path} is missing 'memory_bank'. Rerun cell 8 to rebuild it."
        )
    model.set_memory_bank(memory_bank)
    filtered_state_dict = {key: value for key, value in state_dict.items() if key != "memory_bank"}
    model.load_state_dict(filtered_state_dict, strict=False)
    model.eval()

    return {
        "summary": summary,
        "val_scores_df": val_scores_df,
        "test_scores_df": test_scores_df,
        "threshold_sweep_df": threshold_sweep_df,
        "metrics": metrics,
        "best_sweep": best_sweep,
        "output_dir": run_output_dir,
        "eval_dir": eval_dir,
        "model": model,
        "artifact_info": {"plots_dir": eval_dir / "plots", "loaded_from_artifacts": True},
    }

def get_selected_variant_context() -> tuple[str, dict[str, object], bool]:
    candidate_name = globals().get("selected_variant_name")
    if candidate_name is not None and "variant_outputs" in globals() and candidate_name in variant_outputs:
        return str(candidate_name), variant_outputs[str(candidate_name)], False

    resolved_name = resolve_selected_variant_name_from_artifacts()
    if "variant_outputs" in globals() and resolved_name in variant_outputs:
        return resolved_name, variant_outputs[resolved_name], False

    selected_variant = load_selected_variant_from_artifacts(resolved_name)
    return resolved_name, selected_variant, True

In [ ]:
base_output_dir.mkdir(parents=True, exist_ok=True)

sweep_rows = []
variant_outputs = {}
ranking_metrics = [AUTO_SELECT_METRIC, *[metric for metric in ["f1", "auroc", "auprc"] if metric != AUTO_SELECT_METRIC]]

for variant in PATCHCORE_SWEEP:
    variant_name = str(variant["name"])
    run_output_dir = base_output_dir / variant_name
    eval_dir = run_output_dir / "evaluation"
    run_output_dir.mkdir(parents=True, exist_ok=True)
    eval_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n=== PatchCore variant: {variant_name} ===")
    model = build_patchcore_model(reduction=variant["reduction"], topk_ratio=variant["topk_ratio"])
    memory_info = get_memory_bank_info(int(variant["memory_bank_size"]))
    model.set_memory_bank(memory_info["memory_bank"].to(device))

    run_config = copy.deepcopy(config)
    run_config["run"]["output_dir"] = run_output_dir.relative_to(REPO_ROOT).as_posix()
    run_config["model"]["memory_bank_size"] = int(variant["memory_bank_size"])
    run_config["model"]["reduction"] = str(variant["reduction"])
    run_config["model"]["topk_ratio"] = float(variant["topk_ratio"])

    checkpoint = {
        "model_state_dict": model.state_dict(),
        "config": run_config,
        "memory_bank_size": int(model.memory_bank.shape[0]),
        "feature_dim": int(model.feature_dim),
        "patches_per_image": int(model.patches_per_image),
        "backbone_type": str(base_model_config.get("backbone_type", "resnet18")),
    }
    torch.save(checkpoint, run_output_dir / "best_model.pt")
    torch.save(checkpoint, run_output_dir / "last_model.pt")

    val_scores_df = collect_scores(model, val_loader)
    test_scores_df = collect_scores(model, test_loader)

    val_normal_scores = val_scores_df.loc[val_scores_df["is_anomaly"] == 0, "score"]
    threshold = float(val_normal_scores.quantile(0.95))
    labels = test_scores_df["is_anomaly"].to_numpy()
    scores = test_scores_df["score"].to_numpy()

    metrics = summarize_threshold_metrics(labels, scores, threshold)
    threshold_sweep_df, best_sweep = sweep_threshold_metrics(labels, scores)

    summary = {
        "name": variant_name,
        "memory_bank_size": int(model.memory_bank.shape[0]),
        "memory_subset_images": int(memory_info["memory_subset_images"]),
        "patches_per_image": int(memory_info["patches_per_image"]),
        "feature_dim": int(memory_info["feature_dim"]),
        "reduction": str(variant["reduction"]),
        "topk_ratio": float(variant["topk_ratio"]),
        "threshold": threshold,
        "precision": float(metrics["precision"]),
        "recall": float(metrics["recall"]),
        "f1": float(metrics["f1"]),
        "auroc": float(metrics["auroc"]),
        "auprc": float(metrics["auprc"]),
        "best_sweep_threshold": float(best_sweep["threshold"]),
        "best_sweep_precision": float(best_sweep["precision"]),
        "best_sweep_recall": float(best_sweep["recall"]),
        "best_sweep_f1": float(best_sweep["f1"]),
        "predicted_anomalies": int(metrics["predicted_anomalies"]),
        "output_dir": run_output_dir.relative_to(REPO_ROOT).as_posix(),
        "plots_dir": (eval_dir / "plots").relative_to(REPO_ROOT).as_posix(),
    }

    val_scores_df.to_csv(eval_dir / "val_scores.csv", index=False)
    test_scores_df.to_csv(eval_dir / "test_scores.csv", index=False)
    threshold_sweep_df.to_csv(eval_dir / "threshold_sweep.csv", index=False)
    with (run_output_dir / "summary.json").open("w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2)

artifact_info = save_patchcore_variant_artifacts(
    model=model,
    variant_name=variant_name,
    eval_dir=eval_dir,
    val_scores_df=val_scores_df,
    test_scores_df=test_scores_df,
    threshold_sweep_df=threshold_sweep_df,
    threshold=threshold,
    best_sweep_threshold=float(best_sweep["threshold"]),
    include_umap=False,
)

print(summary)
sweep_rows.append(summary)
variant_outputs[variant_name] = {
    "summary": summary,
    "val_scores_df": val_scores_df,
    "test_scores_df": test_scores_df,
    "threshold_sweep_df": threshold_sweep_df,
    "metrics": metrics,
    "best_sweep": best_sweep,
    "output_dir": run_output_dir,
    "eval_dir": eval_dir,
    "model": model,
    "artifact_info": artifact_info,
}

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

sweep_results_df = pd.DataFrame(sweep_rows).sort_values(ranking_metrics, ascending=False).reset_index(drop=True)
sweep_results_df.to_csv(base_output_dir / "patchcore_sweep_results.csv", index=False)

if SELECTED_VARIANT_NAME is not None:
    selected_variant_name = SELECTED_VARIANT_NAME
else:
    selected_variant_name = str(sweep_results_df.iloc[0]["name"])

selected_row = sweep_results_df.loc[sweep_results_df["name"] == selected_variant_name].iloc[0].to_dict()
with (base_output_dir / "patchcore_sweep_summary.json").open("w", encoding="utf-8") as handle:
    json.dump(
        {
            "selected_variant_name": selected_variant_name,
            "auto_select_metric": AUTO_SELECT_METRIC,
            "ranking_metrics": ranking_metrics,
            "selected_row": selected_row,
            "results": sweep_rows,
        },
        handle,
        indent=2,
    )

print(f"Selected variant for downstream plots: {selected_variant_name}")
display(sweep_results_df)

In [ ]:
selected_variant_name, selected_variant, loaded_from_artifacts = get_selected_variant_context()
output_dir = selected_variant["output_dir"]
summary = selected_variant["summary"]
val_scores_df = selected_variant["val_scores_df"]
test_scores_df = selected_variant["test_scores_df"]
threshold_sweep_df = selected_variant["threshold_sweep_df"]
metrics = selected_variant["metrics"]
best_sweep = selected_variant["best_sweep"]
threshold = float(summary["threshold"])

metrics_df = pd.DataFrame(
    [
        {"metric": "precision", "value": metrics["precision"]},
        {"metric": "recall", "value": metrics["recall"]},
        {"metric": "f1", "value": metrics["f1"]},
        {"metric": "auroc", "value": metrics["auroc"]},
        {"metric": "auprc", "value": metrics["auprc"]},
        {"metric": "threshold", "value": threshold},
    ]
)

display(metrics_df)
display(pd.DataFrame(metrics["confusion_matrix"], index=["true_normal", "true_anomaly"], columns=["pred_normal", "pred_anomaly"]))
print(f"Selected variant: {selected_variant_name}")
if loaded_from_artifacts:
    print(f"Reloaded saved checkpoint + score artifacts from disk: {output_dir}")
print(f"Best sweep threshold: {best_sweep['threshold']:.6f} | precision={best_sweep['precision']:.4f}, recall={best_sweep['recall']:.4f}, f1={best_sweep['f1']:.4f}")

In [ ]:
selected_variant_name, selected_variant, loaded_from_artifacts = get_selected_variant_context()
output_dir = selected_variant["output_dir"]
eval_dir = selected_variant["eval_dir"]
summary = selected_variant["summary"]
val_scores_df = selected_variant["val_scores_df"]
test_scores_df = selected_variant["test_scores_df"]
threshold_sweep_df = selected_variant["threshold_sweep_df"]
metrics = selected_variant["metrics"]
best_sweep = selected_variant["best_sweep"]
threshold = float(summary["threshold"])

if loaded_from_artifacts:
    print(f"Fresh-kernel reload succeeded from {output_dir}")

if GENERATE_SELECTED_UMAP:
    print(f"Generating embedding artifacts for selected variant: {selected_variant_name}")
    selected_artifacts = save_patchcore_variant_artifacts(
        model=selected_variant["model"],
        variant_name=selected_variant_name,
        eval_dir=eval_dir,
        val_scores_df=val_scores_df,
        test_scores_df=test_scores_df,
        threshold_sweep_df=threshold_sweep_df,
        threshold=threshold,
        best_sweep_threshold=float(best_sweep["threshold"]),
        include_umap=True,
    )
    print({
        "plots_dir": str(selected_artifacts["plots_dir"]),
        "umap_points_path": str(selected_artifacts.get("umap_points_path")) if selected_artifacts.get("umap_points_path") is not None else None,
    })
else:
    print("Skipping UMAP generation because GENERATE_SELECTED_UMAP=False")

In [ ]:
plot_df = sweep_results_df.copy()
plot_df["label"] = plot_df["name"] + "\n" + plot_df["reduction"] + ", mb=" + plot_df["memory_bank_size"].astype(str)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].barh(plot_df["label"], plot_df["f1"], color="#277da1")
axes[0].set_title("PatchCore ResNet18 Sweep: Validation-Threshold F1")
axes[0].set_xlabel("F1")
axes[0].invert_yaxis()

axes[1].barh(plot_df["label"], plot_df["auroc"], color="#90be6d")
axes[1].set_title("PatchCore ResNet18 Sweep: AUROC")
axes[1].set_xlabel("AUROC")
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(output_dir / "patchcore_sweep_barplots.png", dpi=200, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(val_scores_df["score"], bins=30, alpha=0.8, color="#4d908e")
axes[0].axvline(threshold, color="red", linestyle="--", label=f"threshold={threshold:.4f}")
axes[0].set_title(f"Validation Normal Score Distribution\n{selected_variant_name}")
axes[0].legend()

axes[1].hist(test_scores_df[test_scores_df["is_anomaly"] == 0]["score"], bins=30, alpha=0.7, label="normal")
axes[1].hist(test_scores_df[test_scores_df["is_anomaly"] == 1]["score"], bins=30, alpha=0.7, label="anomaly")
axes[1].axvline(threshold, color="red", linestyle="--", label=f"threshold={threshold:.4f}")
axes[1].set_title(f"Test Score Distribution\n{selected_variant_name}")
axes[1].legend()

plt.tight_layout()
plt.savefig(eval_dir / "plots" / "selected_variant_histograms.png", dpi=200, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(threshold_sweep_df["threshold"], threshold_sweep_df["precision"], label="precision")
plt.plot(threshold_sweep_df["threshold"], threshold_sweep_df["recall"], label="recall")
plt.plot(threshold_sweep_df["threshold"], threshold_sweep_df["f1"], label="f1")
plt.axvline(threshold, color="red", linestyle="--", label=f"validation threshold = {threshold:.4f}")
plt.axvline(best_sweep["threshold"], color="green", linestyle=":", label=f"best sweep threshold = {best_sweep['threshold']:.4f}")
plt.title(f"Threshold Sweep on Test Split\n{selected_variant_name}")
plt.xlabel("Anomaly-score threshold")
plt.ylabel("Metric value")
plt.legend()
plt.tight_layout()
plt.savefig(eval_dir / "plots" / "selected_variant_threshold_sweep.png", dpi=200, bbox_inches="tight")
plt.show()

## Failure Analysis

This section attaches the selected ResNet18 PatchCore scores to the test metadata and highlights the main false-positive and false-negative patterns.

In [ ]:
analysis_df = test_dataset.metadata.reset_index(drop=True).copy()
analysis_df["score"] = test_scores_df.reset_index(drop=True)["score"]
analysis_df["predicted_anomaly"] = (analysis_df["score"] > threshold).astype(int)

analysis_df["error_type"] = "tn"
analysis_df.loc[(analysis_df["is_anomaly"] == 0) & (analysis_df["predicted_anomaly"] == 1), "error_type"] = "fp"
analysis_df.loc[(analysis_df["is_anomaly"] == 1) & (analysis_df["predicted_anomaly"] == 0), "error_type"] = "fn"
analysis_df.loc[(analysis_df["is_anomaly"] == 1) & (analysis_df["predicted_anomaly"] == 1), "error_type"] = "tp"

error_summary_df = (
    analysis_df.groupby("error_type")
    .agg(count=("error_type", "size"), mean_score=("score", "mean"))
    .reindex(["tp", "fn", "fp", "tn"])
)

defect_recall_df = (
    analysis_df[analysis_df["is_anomaly"] == 1]
    .groupby("defect_type")
    .agg(count=("defect_type", "size"), detected=("predicted_anomaly", "sum"), mean_score=("score", "mean"))
    .sort_values(["detected", "count"], ascending=[False, False])
)
defect_recall_df["recall"] = defect_recall_df["detected"] / defect_recall_df["count"]

print(f"Failure analysis variant: {selected_variant_name}")
display(error_summary_df)
display(defect_recall_df)
analysis_df.head()

In [ ]:
from pathlib import Path
import pandas as pd

umap_csv = eval_dir / "plots" / "embedding_umap_points.csv"
umap_df = pd.read_csv(umap_csv)

display(umap_df.head())
print(umap_df["split_label"].value_counts(dropna=False))

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

# Use only test points for overlap analysis
test_df = umap_df[umap_df["split_label"].isin(["test_normal", "test_anomaly"])].copy()

X = test_df[["umap_1", "umap_2"]].to_numpy()
y = test_df["is_anomaly"].to_numpy()

k = 15  # local neighborhood size
nbrs = NearestNeighbors(n_neighbors=k + 1, metric="euclidean")
nbrs.fit(X)

distances, indices = nbrs.kneighbors(X)

# skip self-neighbor at column 0
neighbor_idx = indices[:, 1:]
neighbor_labels = y[neighbor_idx]

# fraction of anomaly neighbors around each point
test_df["anomaly_neighbor_ratio"] = neighbor_labels.mean(axis=1)

# fraction of normal neighbors around each point
test_df["normal_neighbor_ratio"] = 1.0 - test_df["anomaly_neighbor_ratio"]

display(test_df.head())

In [ ]:
# anomaly points whose local neighborhood is mostly normal
embedded_anomaly_threshold = 0.8

embedded_anomalies = test_df[
    (test_df["is_anomaly"] == 1) &
    (test_df["normal_neighbor_ratio"] >= embedded_anomaly_threshold)
].copy()

print(f"Total test anomalies: {(test_df['is_anomaly'] == 1).sum()}")
print(f"Embedded anomalies (>= {embedded_anomaly_threshold:.0%} normal neighbors): {len(embedded_anomalies)}")

embedded_fraction = len(embedded_anomalies) / max((test_df["is_anomaly"] == 1).sum(), 1)
print(f"Embedded anomaly fraction: {embedded_fraction:.3f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 7))

# all test normals
normals = test_df[test_df["is_anomaly"] == 0]
plt.scatter(
    normals["umap_1"], normals["umap_2"],
    s=10, alpha=0.18, label="test_normal"
)

# all test anomalies
anoms = test_df[test_df["is_anomaly"] == 1]
plt.scatter(
    anoms["umap_1"], anoms["umap_2"],
    s=14, alpha=0.45, label="test_anomaly"
)

# embedded anomalies highlighted
plt.scatter(
    embedded_anomalies["umap_1"], embedded_anomalies["umap_2"],
    s=28, alpha=0.95, marker="x", label="embedded_anomaly"
)

plt.title("UMAP with Embedded Anomalies Highlighted")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.legend()
plt.tight_layout()
plt.show()